# Parallel Processing With Enhancements
Enhanced version accepts several functions:
* initializer - allows for a function to be called to setup initial conditions
* main - star of the show; the function to do the work; that which the other functions wrap around
* callback - a function to run once main has completed

### As for `concurrent.futures`
I investigated this functionality and found it good in ways, but too limited for what I wanted to do.  
* Futures can only be cancelled if the function called has not been started
* Callback function has no means of knowing the parameters supplied to the called function

### Imports

In [117]:
import time, random

## Setup Logging
Should functions emit messages to one or more of standard output, a log file, etc.

## Functions

### Helper functions

In [118]:
def info(a):
	return f'{a} [Type: {type(a)}]'

### Initializer

In [119]:
def my_init(fn, **kwargs):
    """
    Initialization function to be called immediately prior to invoking the main function.
    Records the start of execution, function called, and passed parameters
    
    fn: Function to be called
    **kwargs: dict of key / value paired parameters to pass to fn
    """
    print(f'Calling: {fn.__name__}({kwargs})')

### main
Various functions to call.  These do a little math and delay for a moment to illustrate threads finishing at different times.

In [ ]:
def fn1(message: str, n: int, **kwargs):
    """
    A sample main function to be called.  
    This one sums the squares of a 1..n and returns the value.
    
    Parameters
    ----------------------------------------------------------------------------
    message: Function to be called

    **kwargs - dict of key / value paired parameters to pass to fn

    Returns
    ----------------------------------------------------------------------------
    None
    """
    
    # Calculate the return value
    ret = 0
    for i in range(1, n + 1):
        ret += i**2

    # Wait a moment
    r = n * random.random()
    time.sleep(r)

    # Print a message and return the value
    print(f'\tDone in {r:.2f} seconds', f'\n\t(random wait factor {n})' if n > 1 else '')
    return ret


def fn2(**kwargs):
    """
    A sample main function to be called.  
    This one returns the n!, if given; otherwise returns 1
    
    message: Function to be called
    **kwargs - dict of key / value paired parameters to pass to fn
    """

    # Catch the given value for n, use 1 if no such parameter
    n = kwargs['n'] if kwargs.__contains__('n') else 1
    
    # Calculate the return value
    ret = 1
    for i in range(1, n + 1):
        ret *= i

    # Wait a moment    
    r = n * random.random()
    time.sleep(r)

    # Print a message and return the value
    print(f'\tDone in {r:.2f} seconds', f'\n\t(random wait factor {n})' if kwargs.__contains__('n') else '')
    return ret

### Callback

In [121]:
def my_callback(fn, **kwargs):
    print(f'Finished: {fn.__name__}({kwargs})\n')

### Parallelize
Accepts a list of functions and the keyword arguments to pass them.  
Invokes a configurable number of these in parallel, invoking the next when a thread becomes available,
eventually calling all in the list.

In [122]:
def parallelize(function_calls: list):
	for fn, kwargs in function_calls:
		my_init(fn, **kwargs)

		try:
			print(f'\tResult: {fn(**kwargs)}')

		except Exception as ex:
			print(type(ex), '\n\t', ex)

		my_callback(fn, **kwargs)

In [124]:
call_sheet = [
	(fn1, {'message':'Summing squares through 4 should be 30...', 'n':4, 'r':30}),
	(fn1, {'message':'Summing squares through 7 should be 140', 'n':7, 'r':140}),
	(fn1, {'message':"Oh, no! I'm gonna fail on missing parameter 'n'", 'r':0}),
	(fn1, {'message':"Oh, no! I'm gonna fail due to negative 'n'", 'n': -2, 'r':0}),
	(fn2, {'dummy':'no delay'}),
	(fn2, {'dummy':'Delay specified', 'n': 7, 'r': 111}),
	]

parallelize(call_sheet)

Calling: fn1({'message': 'Summing squares through 4 should be 30...', 'n': 4, 'r': 30})
	Done in 0.87 seconds 
	(random wait factor 4)
	Result: 30
Finished: fn1({'message': 'Summing squares through 4 should be 30...', 'n': 4, 'r': 30})

Calling: fn1({'message': 'Summing squares through 7 should be 140', 'n': 7, 'r': 140})
	Done in 3.64 seconds 
	(random wait factor 7)
	Result: 140
Finished: fn1({'message': 'Summing squares through 7 should be 140', 'n': 7, 'r': 140})

Calling: fn1({'message': "Oh, no! I'm gonna fail on missing parameter 'n'", 'r': 0})
<class 'TypeError'> 
	 fn1() missing 1 required positional argument: 'n'
Finished: fn1({'message': "Oh, no! I'm gonna fail on missing parameter 'n'", 'r': 0})

Calling: fn1({'message': "Oh, no! I'm gonna fail due to negative 'n'", 'n': -2, 'r': 0})
<class 'ValueError'> 
	 sleep length must be non-negative
Finished: fn1({'message': "Oh, no! I'm gonna fail due to negative 'n'", 'n': -2, 'r': 0})

Calling: fn2({'dummy': 'no delay'})
	Done in